# Healthcare CATE — offline evaluation of a treatment-assignment rule

Mirror of `examples/use_cases/03_healthcare_cate.py`. Estimate the
policy value of a candidate clinical decision rule from retrospective
logs, **before** any RCT. Particularly attentive to overlap violations,
which are the dominant trust risk in healthcare OPE.

👉 [Open in Colab](https://colab.research.google.com/github/dgenio/skdr-eval/blob/main/examples/notebooks/05_healthcare_cate.ipynb)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet skdr-eval scikit-learn

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

import skdr_eval

logs, treatments, _ = skdr_eval.make_synth_logs(n=2000, n_ops=3, seed=23)
skdr_eval.validate_logs(logs)
list(treatments)

In [ ]:
candidates = {
    "RF_rule": RandomForestRegressor(n_estimators=80, max_depth=6, random_state=23),
    "HGB_rule": HistGradientBoostingRegressor(max_iter=80, random_state=23),
}
artifact = skdr_eval.evaluate_sklearn_models(
    logs=logs,
    models=candidates,
    fit_models=True,
    n_splits=2,
    outcome_estimator="hgb",
    random_state=23,
    policy_train="pre_split",
    policy_train_frac=0.8,
)
cols = ["model", "estimator", "V_hat", "ESS", "match_rate", "support_health"]
artifact.report[cols].round(4)

## DR vs. SNDR disagreement

A large gap between the DR and SNDR estimates is the most common
trust failure in healthcare OPE — it usually means thin overlap
is inflating DR variance. Report both values to the clinical team.

In [ ]:
for model_name in candidates:
    rows = artifact.report[artifact.report["model"] == model_name]
    dr = rows[rows["estimator"] == "DR"]["V_hat"].iloc[0]
    sndr = rows[rows["estimator"] == "SNDR"]["V_hat"].iloc[0]
    print(f"{model_name}: |DR - SNDR| = {abs(dr - sndr):.3f}")

Per-subgroup CATE slicing (e.g., by age band or comorbidity) is
tracked in roadmap issue #87.